# HealthCare Platform — AI Component Technical Report
### Medical Language Model Integration

---

## Table of Contents
1. [Introduction](#1-introduction)
2. [AI Model Overview](#2-ai-model-overview)
3. [Benchmark Performance](#3-benchmark-performance)
4. [System Integration and Usage](#4-system-integration-and-usage)
5. [System Demonstration](#5-system-demonstration)
6. [Conclusion](#6-conclusion)

---
## 1. Introduction

The HealthCare platform connects patients with the right medical specialist through an AI-powered assistant. The assistant analyzes patient symptoms in natural language, provides an initial clinical assessment, and routes the patient to the most relevant specialty available in the platform's doctor network.

This report covers the AI model used, its training background, benchmark results, and how it is configured and integrated within the system.

---
## 2. AI Model Overview

The model integrated in the platform is a large language model (LLM) fine-tuned specifically for medical and clinical applications. It is built on a state-of-the-art transformer architecture and fine-tuned on medical literature, clinical notes, and structured medical Q&A datasets.

The variant used is the **4-billion parameter text-to-text version**, selected for its balance between clinical accuracy and computational efficiency. It processes text input and returns text output only — no images or other media.

---
## 3. Benchmark Performance

The model was evaluated on established medical benchmarks covering clinical reasoning, abstract-level decision making, and regional medicine:

| Benchmark | Task Type | MedGemma 4B Score |
|---|---|---|
| MedQA (USMLE) | Clinical Vignette MCQ | 63.4% |
| PubMedQA | Abstract Yes/No/Maybe | 74.2% |
| MedMCQA | Medical Entrance Exam MCQ | 60.1% |
| MedXpertQA | Expert multi-step reasoning | 44.1% |
| AfriMed-QA | Tropical & Regional Medicine | 55.8% |

These results reflect strong performance across diverse medical domains. In the HealthCare platform, the model is not used as a standalone diagnostic engine — it acts as a guided assistant that assesses symptoms and routes patients to the correct specialty.

---
## 4. System Integration and Usage

### 4.1 Role in the Platform

The model performs two functions simultaneously in every patient interaction:

- **Clinical response** — analyzes the patient's symptoms and provides an empathetic initial assessment, asking clarifying questions if needed.
- **Specialty suggestion** — selects the most appropriate medical specialty from the platform's available list, which is then used to query the doctor database filtered by specialty and patient city.

### 4.2 Specialty Injection Flow

The model has no hardcoded knowledge of the platform's specialties. The flow works as follows:

1. The patient sends a message through the app.
2. The platform's **.NET API** fetches the current specialty list from the database.
3. The specialties are appended to the system prompt as a comma-separated string.
4. The full prompt + patient message is sent to the model.
5. The model returns a JSON response with a clinical message and a `suggestedSpecialty`.
6. The platform uses `suggestedSpecialty` to search the doctor database filtered by specialty and patient city.

### 4.3 Doctor Recommendation

The `suggestedSpecialty` value matches exactly a specialty name in the database, enabling a direct query for high-rated doctors in that specialty within the patient's city.

### 4.4 Model Setup


#### Install dependencies

In [ ]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

#### Load Model

The model is loaded with 4-bit quantization to reduce memory usage.

In [ ]:
from transformers import BitsAndBytesConfig
import torch

model_id = "google/medgemma-4b-it"

model_kwargs = dict(
    dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

**Load model with the `pipeline` API**

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model_id, model_kwargs=model_kwargs)

**Load model directly**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_id)

### 4.5 System Prompt

The system prompt below defines the model's behavior. It instructs the model to act as a professional medical assistant, respond only in JSON format, and always select a specialty from the list injected by the .NET API.

In [ ]:
SYSTEM_PROMPT = (
    "You are a professional medical assistant for the HealthCare platform. "
    "Your role is to analyze patient symptoms, provide an initial diagnosis, "
    "and guide them to the correct medical specialty. "
    "Rules: "
    "1. Be empathetic and professional. "
    "2. Do NOT include disclaimers about being an AI or state that you cannot provide a diagnosis; "
    "instead, provide a direct initial assessment and guidance. "
    "3. You are encouraged to provide an initial diagnosis and ask necessary clarifying questions. "
    "4. If the case is clear (e.g., pregnancy), provide a detailed and comprehensive response "
    "with relevant advice, but remain concise (do not write an entire page). "
    "5. Suggest the most relevant medical specialty ONLY from the provided list. "
    "The specialty must match EXACTLY (case-sensitive). "
    "6. Do NOT translate specialty names. "
    "7. If no clear specialty can be determined, return null for suggestedSpecialty. "
    "8. ALWAYS respond in valid JSON only with no extra text. "
    'Response format: { "message": "your response here", "suggestedSpecialty": "ExactSpecialtyName or null" }'
    # The .NET API appends the specialty list here at runtime:
    # + " Available specialties: Cardiology, Neurology, Orthopedics, ..."
)

### 4.6 Running Inference (Text Only)

The conversation is formatted with the system prompt and the patient message, then sent to the model. The .NET API handles specialty injection and JSON parsing in production — the cell below shows the model interaction directly.

In [ ]:
from IPython.display import Markdown

# In production, the .NET API appends specialties from the database to SYSTEM_PROMPT
# and sends the patient message. Here we show the model interaction directly.
prompt = "I've been feeling a sharp pain in my chest whenever I take a deep breath, and I have a persistent dry cough for 3 days now. It gets worse at night."

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }
]

max_new_tokens = 500

**Run model with the `pipeline` API**

In [ ]:
output = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

**Run the model directly**

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = tokenizer.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---
## 5. System Demonstration

The following examples show real interactions with the AI assistant as integrated in the HealthCare platform. Each shows the patient input, the model's clinical response, and the specialty selected from the database list.

### Example 1 — Chest Pain and Persistent Cough

**Patient input:**

In [ ]:
patient_input_1 = {
    "message": "I've been feeling a sharp pain in my chest whenever I take a deep breath, and I have a persistent dry cough for 3 days now. It gets worse at night."
}

**Model response:**

In [ ]:
response_1 = {
    "message": "Based on your symptoms of sharp chest pain with deep breathing and a persistent dry cough that worsens at night, it's difficult to pinpoint the exact cause without a physical examination and further investigation. However, these symptoms could potentially be related to several conditions, including pleurisy, asthma, or even musculoskeletal issues. It's important to rule out any serious underlying causes.",
    "suggestedSpecialty": "Pulmonology"
}

### Example 2 — Early Pregnancy Symptoms

**Patient input:**

In [ ]:
patient_input_2 = {
    "message": "I just found out I am pregnant. I'm feeling very nauseous in the morning and I have some mild cramping. Is this normal?"
}

**Model response:**

In [ ]:
response_2 = {
    "message": "Congratulations on your pregnancy! Morning sickness and mild cramping are common symptoms in early pregnancy. However, it's important to monitor your symptoms and seek medical advice if they worsen or if you experience any other concerning signs. It's recommended to schedule an appointment with your obstetrician or a family physician specializing in women's health to confirm the pregnancy and discuss your symptoms in detail. They can provide personalized guidance and ensure a healthy pregnancy journey.",
    "suggestedSpecialty": "Obstetrics and Gynecology"
}

### Example 3 — Headache and Dizziness

**Patient input:**

In [ ]:
patient_input_3 = {
    "message": "My head hurts and I feel dizzy since this morning."
}

**Model response:**

In [ ]:
response_3 = {
    "message": "I understand you're experiencing a headache and dizziness. To better assess your condition, could you please provide more details? For example, where is the headache located? Is it a throbbing or a constant pain? Have you had any recent head trauma? What other symptoms are you experiencing, such as nausea, vomiting, vision changes, or neck stiffness?",
    "suggestedSpecialty": "Neurology"
}

---
## 6. Conclusion

The AI model integrated in the HealthCare platform is a medically fine-tuned large language model chosen for its strong benchmark results and professional clinical communication.

Through system prompt engineering and dynamic specialty injection from the database (handled by the .NET API at runtime), the model simultaneously generates a clinical response and returns a structured specialty suggestion — enabling automated, personalized doctor recommendations by specialty and patient city.

This design adapts the model to a real-world healthcare application without modifying the model itself. The specialty list is fully controlled by the database, making the system flexible and maintainable.